# Module B04 — Repeating Work: Loops

## Exercise 5: Positions, without counting them yourself

There is a loop that beginners write in every language, and it looks like this:

```python
for i in range(len(names)):
    print(names[i])
```

It works. It also asks for a run of numbers when what you wanted was a run of
names, and then converts each number back into a name by hand. Every one of
those steps is a place to make a mistake, and one of them has a favourite
mistake all of its own.

This notebook teaches the position machinery honestly, shows you the two tools
that remove the need for most of it, and ends with the bug that catches people
who change a list while walking through it.

| | |
|---|---|
| Time | About 45 minutes |
| You need | This notebook |
| Comes after | Exercise 4, nested loops |

---

## 1. Items have positions, and positions start at zero

Square brackets after a list ask for the item at a position.

```
names = ["ama", "kofi", "yaa", "kwame"]

position:   0      1       2      3
            ^                     ^
            |                     |
        the first             the last, which is len(names) - 1
```

Counting from zero is the same convention as `range(5)` starting at zero, which
is why `range(len(names))` produces exactly the valid positions and no others.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

print("names[0] is", names[0])
print("names[2] is", names[2])
print("len(names) is", len(names))
print("the last position is", len(names) - 1)
print("names[-1] is", names[-1], "which is a shortcut for the last one")

---

## 2. The index-based loop

Now the loop from the opening. It is worth writing out once, because you have to
be able to read it in other people's code.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

for i in range(len(names)):
    print(i, names[i])

`range(len(names))` is `range(4)`, which is 0, 1, 2, 3. Each pass, `names[i]`
turns the number back into a name.

Nothing is wrong with that loop. It is correct. It is also three moving parts
where the job needed one, and the next section is what happens when one of the
three slips.

---

## 3. The off-by-one, with a traceback this time

Exercise 1 showed an off-by-one that gave a wrong answer in silence. When the
number is used as a position, the same mistake is loud instead.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

for i in range(len(names) + 1):
    print(i, names[i])

It printed all four names, then:

```
IndexError: list index out of range
```

**`IndexError`** means you asked for a position the list does not have. There
are four names, at positions 0 to 3. The `+ 1` made the loop ask for position 4,
and there is no position 4.

Note where the traceback appears: after four correct lines. Most of the loop
worked. That is normal for an `IndexError`, and it is why the printed output
above the error is worth reading, because the count of lines tells you where it
gave up.

`len(names)` is the count, and the last position is `len(names) - 1`. Confusing
those two is the whole of this error.

---

## 4. If you do not need the position, do not ask for one

Most of the time the number was never the point.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

for name in names:
    print(name)

One moving part. There is no number to get wrong, no `len` to be one out on, and
no `IndexError` available to you. The line also says what it means: for each
name in names.

Make this your default. Reach for positions when the job genuinely needs them,
and the next three sections are about what to do then.

---

## 5. `enumerate` hands you the position and the item together

When you need both, `enumerate` gives you both, and the loop takes two names
instead of one.

```
for position, name in enumerate(names):
│      │        │             │
│      │        │             └── the list, unchanged
│      │        └── the second name gets the item
│      └── the first name gets the position
└── the keyword
```

Each pass `enumerate` produces a pair, and the two names split the pair between
them. That splitting is called unpacking, and module B05 covers it properly.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

for position, name in enumerate(names):
    print(position, name)

Same output as section 2, and now there is no `range`, no `len`, and no
`names[i]`. The `IndexError` from section 3 cannot be written.

If you use only one name, you get the pair itself rather than the item, which
prints with brackets and is almost never what you wanted. Two names, always.

---

## 6. `enumerate(names, start=1)` for numbering people can read

Positions start at zero. Numbered lists shown to humans start at one.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]

for number, name in enumerate(names, start=1):
    print(str(number) + ".", name)

`start=1` changes the number you are handed. It does not change the list and it
does not change where the items are.

That distinction matters the moment you mix the two. If you take a number from
`enumerate(names, start=1)` and use it as a position, you are one past the item
you meant, and at the end of the list you get an `IndexError`. Use `start` for
display, and if you also need the real position, do not use `start` at all.

---

## 7. `zip` walks two lists in step

Two lists that line up, item for item, are walked together by `zip`.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]
scores = [72, 91, 65, 88]

for name, score in zip(names, scores):
    print(name, "scored", score)

`zip` produces a pair each pass, exactly as `enumerate` does, so it takes two
names in the same way.

This replaces the index loop that was reaching into two lists at once, which is
where the position was doing real work and was also easiest to get wrong.

---

## 8. `zip` stops at the shorter list, and says nothing

This is the one thing about `zip` that will cost you.

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]
scores = [72, 91]

for name, score in zip(names, scores):
    print(name, "scored", score)

print("printed 2 of", len(names), "names, with no error")

Two lines. Yaa and Kwame were dropped, and Python did not mention it.

This is a silent data loss, and it is the reason two lists that are supposed to
line up should be checked rather than trusted. If the lists come from two
different files, two different queries, or two different people, they will one
day be different lengths.

The check is one line, and you have everything you need for it from module B03:

In [ ]:
names = ["ama", "kofi", "yaa", "kwame"]
scores = [72, 91]

if len(names) != len(scores):
    print("mismatch:", len(names), "names but", len(scores), "scores")
else:
    for name, score in zip(names, scores):
        print(name, "scored", score)

---

## 9. Changing a list while you walk through it, run on purpose

Here is the instinct almost everybody has: to drop the bad readings, loop
through and remove them as you go. `.remove(x)` deletes the first item equal to
`x`.

Predict the output before you run it.

In [ ]:
readings = [4, -2, -9, 7, 3]

for reading in readings:
    if reading < 0:
        readings.remove(reading)

print("left with:", readings)

`[4, -9, 7, 3]`. The `-9` is still there.

No traceback, no warning, and a list that looks nearly right. Half the negative
values were removed and half were not. If those were invalid records being
cleaned before a payment run, half of them would have gone through.

Here is why. A `for` loop over a list keeps an internal position and moves it
forward one each pass. Removing an item shuffles everything after it down by
one, into a position the loop has already passed. Watch the position and the
list at the same time.

In [ ]:
readings = [4, -2, -9, 7, 3]

for position, reading in enumerate(readings):
    print("position", position, "sees", reading, "  list is now", readings)
    if reading < 0:
        readings.remove(reading)

print("left with:", readings)

Position 1 saw `-2` and removed it. That shuffled `-9` down into position 1, and
the loop had already finished with position 1. Next pass it went to position 2,
which now held `7`, and `-9` was never looked at again.

Read the printed positions: the loop went 0, 1, 2, 3 and the value `-9` appears
in none of them after the removal. One removal, one item silently skipped.

There is a second symptom in the same bug. The list gets shorter while the loop
is running, so when a removal happens near the end, the loop stops before
reaching the last item.

**Never add to or remove from a list you are looping over.** Build a new one
instead, which is the collecting pattern from exercise 2: start with `[]` above
the loop, `.append` inside it, use it after.

In [ ]:
readings = [4, -2, -9, 7, 3]

kept = []
for reading in readings:
    if reading >= 0:
        kept.append(reading)

print("original:", readings)
print("kept:    ", kept)

The original is untouched, the new list is correct, and the rule that makes it
correct is that nothing changes underneath the loop while it runs.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.** The self-check uses them.

### Task 1

Four index-based loops, written out in the comments. Rewrite each one without
`range(len(...))` and without square brackets.

Use a plain `for` for the one that does not need a position, `enumerate` for the
two that do, and `zip` for the one that walks two lists.

In [ ]:
# ANSWER 1
names = ["ama", "kofi", "yaa"]
scores = [72, 91, 65]

# (a) was: for i in range(len(names)): print(names[i])
print("a:")
for ___ in ___:
    print(___)

# (b) was: for i in range(len(names)): print(i, names[i])
print("b:")
for ___ in ___:
    print(___)

# (c) was: for i in range(len(names)): print(i + 1, names[i])
print("c:")
for ___ in ___:
    print(___)

# (d) was: for i in range(len(names)): print(names[i], scores[i])
print("d:")
for ___ in ___:
    print(___)

### Task 2

Predict **before running**.

In [ ]:
# ANSWER 2
# How many lines will the loop print? ___
# Which names are left out? ___
# Does Python report anything about the ones left out? ___

names = ["ama", "kofi", "yaa", "kwame"]
scores = [72, 91]

for name, score in zip(names, scores):
    print(name, score)

### Task 3

Build an order slip. A numbered list of the products, then each product beside
its price, then the total.

In [ ]:
# ANSWER 3
products = ["rice", "beans", "oil"]
prices = [12.50, 8.00, 21.75]

print("Order")
print("-----")
for ___ in enumerate(products, start=___):
    print(___)

print()
print("With prices")
print("-----------")

total = 0
for ___ in zip(products, prices):
    print(___)
    total += ___

print("total:", total)

### Task 4

The cell below has the section 9 bug. Run it as it stands and note what survives,
then replace it with a version that builds a new list, and answer the question.

In [ ]:
# ANSWER 4
readings = [4, -2, -9, 7, -3, -1]

for reading in readings:
    if reading < 0:
        readings.remove(reading)

print("result:", readings)

# Which negative values survived, and why did the loop miss them? ___

### Task 5

`for i in range(len(items))` is not always wrong. Name one job where you would
still write it, and write that loop below.

If nothing comes to mind, here is a hint that is not the answer: think about a
loop that needs to look at an item and the one after it.

In [ ]:
# ANSWER 5
values = [10, 20, 30, 40]

# A loop that genuinely needs the position.
___

when_an_index_loop_is_still_right = "___"

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 6, the loops that never stop."

a1, a2, a3, a4, a5 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                      _answer("# ANSWER 3"), _answer("# ANSWER 4"),
                      _answer("# ANSWER 5"))

results = [
    check(a1.count("___") == 0 and "range(len(" not in a1.replace(" ", ""),
          "Task 1: all four loops rewritten with no range(len(...)) left"),
    check("enumerate(" in a1 and "zip(" in a1,
          "Task 1: enumerate used for the numbered loops and zip for the paired one"),
    check("start=1" in a1.replace(" ", "") or "start = 1" in a1,
          "Task 1c: numbering starts at one"),
    check("print? ___" not in a2 and a2.count("___") == 0,
          "Task 2: you predicted the zip result before running"),
    check(a3.count("___") == 0 and "enumerate(" in a3 and "zip(" in a3
          and ("total +=" in a3 or "total = total +" in a3),
          "Task 3: the order slip uses enumerate, zip, and a running total"),
    check(".remove(" not in a4 and ".append(" in a4,
          "Task 4: the removing loop is gone and a new list is built"),
    check("miss them? ___" not in a4, "Task 4: you explained what the loop skipped"),
    check("for " in a5 and a5.count("___") == 0,
          "Task 5: an index loop written and its job named"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- Positions start at zero, so the last one is `len(items) - 1`, and
  `range(len(items))` produces exactly the valid positions.
- The index-based loop works and is three moving parts where one would do.
- Asking for a position the list does not have is an `IndexError`, and it
  arrives after the correct output rather than instead of it.
- If you do not need the position, loop over the items and the mistake becomes
  unwritable.
- `enumerate(items)` hands you the position and the item as a pair, taken by two
  names.
- `enumerate(items, start=1)` changes the number reported, not the position. Use
  it for display only.
- `zip(a, b)` walks two lists in step, and stops at the shorter one without
  saying anything, which is a silent data loss worth a length check.
- Removing from a list while looping over it skips items and ends early. Build a
  new list instead.

## Before you move on

- [ ] You read an `IndexError` and can say the difference between `len(items)`
      and the last position.
- [ ] You rewrote an index loop with `enumerate` and one with `zip`.
- [ ] You ran the removing loop and can explain why `-9` survived.
- [ ] You can name one job that still wants an index loop.

**Next:** exercise 6, where three loops never stop at all, shown under a cap so
that nothing here can hang.